# Item 12 — Experimento sem RAG × com RAG no Colab

No VS Code, selecione **Select Kernel → Colab** e um servidor com GPU; execute todas as células. A sessão do chat não controla a seleção/autenticação da extensão.

Este notebook inclui os dez contextos recuperados no computador local e um snapshot do código. O runtime remoto não precisa enxergar o repositório nem o dataset local. A recuperação foi feita por BM25 + semântica → RRF ajustado em dev → reranking Top-50 → Top-5 integral. O manifesto incluído registra essa preparação.

Serão geradas 20 respostas em inglês: A sem documentos, B com documentos, mesma Qwen2.5-1.5B-Instruct, parâmetros, seed 42 e limite de 512 novos tokens. Cada chamada é independente, sem histórico ou cache de conversa compartilhado. A geração usa greedy; isso favorece repetibilidade, mas dispositivos/versões diferentes podem produzir respostas diferentes.

Os resultados abaixo só existem após a execução. Avaliação humana e análise dos itens 13–14 ficam pendentes. O Colab pode interromper o runtime; salve os resultados antes de encerrá-lo.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Selecione um runtime Colab com GPU antes de continuar.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
%pip install -q "transformers==4.57.6" "accelerate==1.15.0"


In [ ]:
import base64, hashlib, io, os, sys, zipfile
from pathlib import Path

payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'f22d773f41b3875956564e71a0a05b0468c15f98734c86339e1600534139ee55'
project = Path('/content/ri_rag_item12')
project.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(project)
os.chdir(project)
sys.path.insert(0, str(project))
print('Snapshot verificado e extraído em', project)


In [ ]:
import json
from ri_project.rag import RAGConfig, LocalCausalGenerator
from ri_project.rag_experiment import run_paired_experiment

bundle = json.loads(Path('Experiments/rag_experiment/contexts.json').read_text())
print('Consultas oficiais:', bundle['query_ids'])
config = RAGConfig(revision='989aa7980e4cf806f80c7fef2b1adb7bc71aa306', temperature=0.0, max_new_tokens=512, seed=42)
generator = LocalCausalGenerator(config, device='cuda')
print('Manifesto:', generator.manifest())
output_dir = Path('Experiments/rag_experiment_gpu')
answers = run_paired_experiment(bundle, generator, output_dir)


In [ ]:
from IPython.display import Markdown, display
display(Markdown((output_dir / 'comparison.md').read_text()))
print(json.loads((output_dir / 'run_status.json').read_text()))


In [ ]:
import shutil
from IPython.display import FileLink, display
# Inclui contextos originais junto das respostas para auditoria local.
shutil.copyfile('Experiments/rag_experiment/contexts.json', output_dir / 'contexts.json')
archive_path = shutil.make_archive('rag_item12_resultados_gpu', 'zip', output_dir)
display(FileLink(Path(archive_path).name))
print('No VS Code, também pode baixar o ZIP pelo explorador Colab → Contents → Download.')
